In [11]:
import datetime as dt
from datetime import datetime, timedelta
import dask
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import xarray as xr
from glob import glob
from time import time
import warnings

import pandas as pd

warnings.simplefilter("ignore")

# Set Parameters

# Plots
- Maps
    - [ ] Monats Aufenthaltswahrscheinlichkeiten
    - [ ] Jahres Aufenthaltswahrscheinlichkeiten
    - [ ] Linen zwischen Stationen
- Connectivitätsmatrix
    - [ ] Monats Averages (logscale)
    - [ ] Jahres Averages (logscale)
    - [ ] Einzelne Jahre

In [21]:
# Parameters
site_counter = np.arange(16)
years = np.arange(2016,2025)
year = 2016
nbins = 100
isPapermill = True
start_site_counter = 10
samples = [
    "Speicherkoog", "KB03", "SW08",
    "Wilhelmshaven", "GB96", "H21",
    "Boknis_Eck", "Gdynia", "Riga",
    "IU7_c", "LL17_c", "Finland",
    "BB23", "BB36", "Speicherkoog_Außen",
    "Wilhelmshaven_Außen",
]
lats = [
    54.0929, 54.692, 54.413,
    53.513, 56.54, 54.666,
    54.5169, 54.5829, 57.3911,
    59.489, 59.02, 59.77,
    55.1751, 54.575, 54.11,
    53.637,
]
lons = [
    8.9487, 10.1035, 11.617,
    8.149, 19.5809, 13.0224,
    10.034, 18.6042, 23.8588,
    21.2021, 21.0478, 23.2663,
    15.4352, 16.001, 8.5,
    8.15,
]

number_sites = len(samples)
number_years = len(years)
dist = 20

# Functions

In [22]:
def connectivity_function(
    ds=None,
    dist=None,
):
    # Mean radius of the Earth in km
    R = 6371  

    # Stations (targets)
    obs_lon_deg = ds.lon
    obs_lat_deg = ds.lat
    obs_lon_km = R * np.cos(np.radians(obs_lat_deg)) * np.radians(obs_lon_deg)
    obs_lat_km = R * np.radians(obs_lat_deg)

    # Particles
    target_lon_deg = xr.DataArray(lons)
    target_lat_deg = xr.DataArray(lats)
    target_lon_km = R * np.cos(np.radians(target_lat_deg)) * np.radians(target_lon_deg)
    target_lat_km = R * np.radians(target_lat_deg)

    # Calculate distance to target
    distance_to_target = (
        ((obs_lon_km - target_lon_km) ** 2) + ((obs_lat_km - target_lat_km) ** 2)
    ) ** 0.5

    # Return number of particles within a certain distance from target
    return (distance_to_target <= dist).sum(dim=["obs", "trajectory"]).compute().values
    # return distance_to_target

In [23]:
trajectory_path = "/gxfs_work/geomar/smomw597/2025_copepods/output/Trajectories/"
out_path = "/gxfs_work/geomar/smomw597/2025_copepods/output/PelagicConnectivity/"

In [24]:
ds_trajectories = xr.Dataset()
monthly_connectivity_matrices = {}
export_vars = ["lat", "lon", "time", "S", "T", "eta", "z", "h0", "age_sec"]
export_vars = ["lat", "lon", "time", "age_sec"]
export_vars = ["lat", "lon", "age_sec"]
if isPapermill == False:
    month_split = [None, 4,9,13,17,22]
else:
    month_split = [None, 30000,61000,92000,122000,153000]
expand_dims_dict = {"site":[start_site_counter], "year":[year],}

In [25]:
lonbins = np.arange(8, 12.25, 0.25)
latbins = np.arange(52, 57.25, 0.25)
print(latbins.shape, lonbins.shape)

(21,) (17,)


In [26]:
lonbins = np.arange(0, 30.25, 0.25)
latbins = np.arange(52, 62.25, 0.25)
# lonbins = np.arange(0, 30.25, 0.5)
# latbins = np.arange(52, 62.25, 0.5)
print(latbins.shape, lonbins.shape)
lonbins_da = (lonbins[1:]+lonbins[:-1])/2
latbins_da = (latbins[1:]+latbins[:-1])/2

(41,) (121,)


In [27]:
trajectories = np.arange(0, 153001, 1000)
# trajectories = np.arange(150000, 153001, 1000)
trajectories = np.arange(100000, 110001, 1000)
# for day_id, day in enumerate(days[1:]):
#     print(day-1000,day)

In [43]:
year = 2024
for s in np.arange(16):
    try:
        file = Path(
            trajectory_path +
            f"PPmill_Nested_{year}0601-{year}1101_dt15min_site{site:02d}_d0m-25m_N1000_seed123.zarr"
        )
        xr.open_zarr(file)
        print("yes",s)
    except:
        print("no",s)

no 0
no 1
no 2
no 3
no 4
no 5
no 6
no 7
no 8
no 9
no 10
no 11
no 12
no 13
no 14
no 15


In [33]:
lonbins_da = (lonbins[1:]+lonbins[:-1])/2
latbins_da = (latbins[1:]+latbins[:-1])/2

site = 3
age_data_arrays = []
age_arrays = []
data_arrays = []
# for site in np.array([1, 2, 6,]):
# for site in np.arange(16):
file = Path(
    trajectory_path +
    f"PPmill_Nested_{year}0601-{year}1101_dt15min_site{site:02d}_d0m-25m_N1000_seed123.zarr"
)
ds_trajectories = xr.open_zarr(file)
daily_arrays = []
for traj in trajectories[:-1]:
    start_day = int(np.floor(traj/1000))
    ds_bins = (ds_trajectories
        # .isel(trajectory=slice(traj,traj+1000))
        .isel(trajectory=traj)
        # .get(["lat", "lon", "age_sec", "time"])
        .get(["lat", "lon", "age_sec"])
        .groupby_bins(
            "age_sec", 
            np.arange(0,29*(60*60*24),(60*60*24)),
            labels=np.arange(1,29)
        )
    )
    age_arrays = []
    for age in np.arange(1,29):
    # for age in np.arange(1,5):
        try:
            test_lons_array = ds_bins[age].lon.values
            test_lats_array = ds_bins[age].lat.values
            hist, xedges, yedges = np.histogram2d(
                test_lons_array, test_lats_array,
                bins=(lonbins, latbins),
            )
            da = xr.DataArray(
                hist,
                dims={
                    "lon":lonbins_da,
                    "lat":latbins_da,
                },
                coords={
                    "lon":lonbins_da,
                    "lat":latbins_da,
                },
                name=f"{age}-{site}-{start_day}",
            ).expand_dims({
                "site":[site],
                "age": [age],
                "start_day": [start_day],
                # "time": [ds_bins[age].time.values[0]],
                }
            )
            age_arrays.append(da)
            # print(f"s: {site}", f"start_day: {start_day}", f"age: {age}")
        except:
            print("no","s",site, "start_day",start_day, "age:",age)
            break
    try:
        print(f"s: {site}", f"start_day: {start_day}")
        age_data = xr.concat(age_arrays, dim="age")
        daily_arrays.append(age_data)
    except:
        print("no")
        break
# try:
daily_data = xr.concat(daily_arrays, dim="start_day")
print(daily_data.nbytes/1e9)
daily_data.to_netcdf(
    Path(out_path+f"ds_pelagicConnectivity_{year}_s{site:02d}.nc")
)
        # data_arrays.append(daily_data)
#     except:
#         print("no")
#         break
# ds = xr.concat(data_arrays, dim="site")
# ds.nbytes/1e9

FileNotFoundError: No such file or directory: '/gxfs_work/geomar/smomw597/2025_copepods/output/Trajectories/PPmill_Nested_20160601-20161101_dt15min_site03_d0m-25m_N1000_seed123.zarr'

In [29]:
# ds.to_netcdf(
#     Path(out_path_hist+f"NewConnectivity/ds_pelagicConnectivity_{year}.nc")
# )

In [ ]:
# cm = np.zeros((16,16))
# for start_day in range(len(ds.start_day)):
#     for age in range(28):
#         for i in range(16):
#             for j in range(16):
#                 try:
#                     val = ds.isel(site=i, age=age, start_day=start_day).where(
#                         (
#                             (ds.isel(site=i, age=age, start_day=start_day) != 0) &
#                             (ds.isel(site=j, age=age, start_day=start_day) != 0)
#                         )
#                     ).sum()
#                     cm[i,j] += val
#                 except:
#                     print("no")